# Smart Posture Correction using Blood Flow Optimization

A concise SimVascular-based CFD optimization system for paralyzed patient posture adjustment.

## System Overview
- **Aorta**: Neck/torso influence on cerebral circulation
- **Abdominal Aorta**: Torso/legs influence on organ perfusion  
- **Coronary**: Torso influence on cardiac circulation
- **Optimization**: Iterative posture adjustment for maximum blood flow

In [ ]:
import os
import subprocess
import re
import shutil
from collections import deque
from dataclasses import dataclass
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

AORTA_PROJECT = r'D:\SimVascularProjects\0011_H_AO_H'
ABDOMINAL_PROJECT = r'D:\SimVascularProjects\0029_H_ABAO_H'
CORONARY_PROJECT = r'D:\SimVascularProjects\0073_H_CORO_H'
RESULTS_FOLDER = r'D:\SimVascularProjects\Results'
DEFAULT_RESULTS_DIR = os.path.join(os.getcwd(), 'default_results')

# SIMVASCULAR PATHS - Update these for your installation
MPI_PATH = r'C:\Program Files\Microsoft MPI\Bin\mpiexec.exe'
SVSOLVER_PATH = r'C:\Program Files\SimVascular\svSolver\2022-08-19'
SVSOLVER_EXE = 'svsolver-msmpi-bin.exe'
SVPOST_EXE = 'svpost-bin.exe'

# SIMULATION SETTINGS
N_PROCESSES = 8
SIM_SUBFOLDER = r'Simulations\Project'
SOLVER_FILE = 'solver.inp'
INFLOW_FILE = 'inflow.flow'
ITERATION_DURATION_MIN = 30
MAX_STATIONARY_WINDOW = 3

print("Configuration loaded successfully")
print(f"Aorta Project: {AORTA_PROJECT}")
print(f"SimVascular Path: {SVSOLVER_PATH}")
print(f"MPI Processes: {N_PROCESSES}")

In [ ]:
# Vessel Configuration Classes
@dataclass
class PostureConfig:
    neck: float = 0
    torso: float = 0
    legs: float = 0

    def __post_init__(self):
        self.neck = max(-30, min(45, self.neck))
        self.torso = max(-15, min(60, self.torso))
        self.legs = max(-30, min(30, self.legs))

    def __str__(self):
        return f"Neck:{self.neck:+.0f}° Torso:{self.torso:+.0f}° Legs:{self.legs:+.0f}°"

    def as_tuple(self):
        return (self.neck, self.torso, self.legs)


def posture_distance(p1, p2):
    return abs(p1.neck - p2.neck) + abs(p1.torso - p2.torso) + abs(p1.legs - p2.legs)


class VesselConfig:
    # Aorta Configuration
    AORTA_SURFACES = {2: 'btrunk', 3: 'carotid', 5: 'outflow', 6: 'rt_carotid', 7: 'subclavian'}
    AORTA_RESISTANCE = {2: 12667, 3: 25333, 5: 2171, 6: 25333, 7: 25333}

    # Abdominal Aorta Configuration
    ABDOMINAL_SURFACES = {
        2: 'SMA', 3: 'hepatic', 5: 'left_external_iliac', 6: 'left_internal_iliac',
        7: 'left_renal', 8: 'right_external_iliac', 9: 'right_internal_iliac',
        10: 'right_renal', 11: 'splenic'
    }
    ABDOMINAL_RESISTANCE = {
        2: 50666, 3: 152000, 5: 20266, 6: 76000, 7: 50666,
        8: 20266, 9: 76000, 10: 50666, 11: 152000
    }

    # Coronary Configuration
    CORONARY_SURFACES = {
        2: 'aorta_outlet', 4: 'lca_br1', 5: 'lca_br2', 6: 'lca_br3',
        7: 'lca_br4', 8: 'lca_br5', 9: 'lca_br6', 10: 'rca_br1',
        11: 'rca_br2', 12: 'rca_br3'
    }
    CORONARY_RESISTANCE = {
        2: 2171, 4: 142857, 5: 200000, 6: 200000, 7: 333333,
        8: 500000, 9: 500000, 10: 150000, 11: 250000, 12: 375000
    }


def _fallback_distribution(surface_map, resistance_map):
    weights = {}
    for sid, name in surface_map.items():
        base_res = resistance_map.get(sid, 1.0)
        weights[name] = 1.0 / base_res if base_res else 1.0
    total = sum(weights.values())
    if not total:
        return {name: 1.0 / len(surface_map) for name in surface_map.values()}
    return {name: value / total for name, value in weights.items()}


def load_baseline_flow_profiles():
    mapping = {
        'aorta': ('Aorta', VesselConfig.AORTA_SURFACES, VesselConfig.AORTA_RESISTANCE),
        'abdominal': ('Abdominal_aorta', VesselConfig.ABDOMINAL_SURFACES, VesselConfig.ABDOMINAL_RESISTANCE),
        'coronary': ('Coronary', VesselConfig.CORONARY_SURFACES, VesselConfig.CORONARY_RESISTANCE)
    }
    profiles = {}
    for key, (folder, surface_map, resistance_map) in mapping.items():
        flows_path = os.path.join(DEFAULT_RESULTS_DIR, folder, 'all_results-flows.txt')
        distribution = {}
        baseline_inflow = 83.3
        try:
            df = pd.read_csv(flows_path, sep='\t')
            row = df.iloc[0].drop(labels='step')
            inflow = abs(row.get('inflow', baseline_inflow))
            if inflow > 0:
                baseline_inflow = inflow
            outlets = {col: abs(val) for col, val in row.items() if col != 'inflow'}
            total_out = sum(outlets.values())
            if total_out > 0:
                distribution = {col: val / total_out for col, val in outlets.items()}
        except FileNotFoundError:
            distribution = {}
        if not distribution:
            distribution = _fallback_distribution(surface_map, resistance_map)
        profiles[key] = {
            'baseline_inflow': baseline_inflow,
            'distribution': distribution
        }
    return profiles


BASELINE_PROFILES = load_baseline_flow_profiles()
FLOW_RATIO_BY_VESSEL = {'aorta': 0.94, 'abdominal': 0.90, 'coronary': 0.86}


@dataclass
class FlowParameters:
    vessel_key: str
    inflow: float
    resistance_multiplier: float
    expected_outflow: float
    outflow_ratio: float
    outlet_distribution: dict


def compute_expected_outflow(vessel_key, inflow, resistance_multiplier):
    base_ratio = FLOW_RATIO_BY_VESSEL.get(vessel_key, 0.9)
    posture_factor = 1.05 - 0.12 * (resistance_multiplier - 1.0)
    ratio = base_ratio * posture_factor
    ratio = max(0.55, min(0.97, ratio))
    return inflow * ratio, ratio


def scale_distribution(distribution, total_outflow):
    return {name: total_outflow * fraction for name, fraction in distribution.items()}


def calculate_flow_parameters(posture, vessel_type):
    vessel_key = vessel_type.lower()
    base_flow = BASELINE_PROFILES.get(vessel_key, {}).get('baseline_inflow', 83.3)

    co_factor = 1.0 + posture.torso * 0.002 + posture.legs * 0.001 - abs(posture.neck) * 0.0005
    inflow = base_flow * max(0.7, min(1.3, co_factor))

    if vessel_key == 'aorta':
        resistance_mult = 1.0 + posture.neck * 0.001 + posture.torso * 0.0005
    elif vessel_key == 'abdominal':
        resistance_mult = 1.0 + posture.torso * 0.001 + posture.legs * 0.002
    elif vessel_key == 'coronary':
        resistance_mult = 1.0 + posture.torso * 0.002
    else:
        resistance_mult = 1.0

    resistance_mult = max(0.7, min(1.3, resistance_mult))

    expected_outflow, ratio = compute_expected_outflow(vessel_key, inflow, resistance_mult)
    outlet_distribution = BASELINE_PROFILES.get(vessel_key, {}).get('distribution', {})
    if not outlet_distribution:
        if vessel_key == 'aorta':
            outlet_distribution = _fallback_distribution(VesselConfig.AORTA_SURFACES, VesselConfig.AORTA_RESISTANCE)
        elif vessel_key == 'abdominal':
            outlet_distribution = _fallback_distribution(VesselConfig.ABDOMINAL_SURFACES, VesselConfig.ABDOMINAL_RESISTANCE)
        else:
            outlet_distribution = _fallback_distribution(VesselConfig.CORONARY_SURFACES, VesselConfig.CORONARY_RESISTANCE)

    return FlowParameters(vessel_key, inflow, resistance_mult, expected_outflow, ratio, outlet_distribution)


print("Vessel configuration classes loaded")

In [ ]:
# SimVascular Execution Functions
def modify_inflow_file(project_path, inflow_rate):
    """Update inflow.flow file with new inflow rate"""
    sim_dir = os.path.join(project_path, SIM_SUBFOLDER)
    inflow_path = os.path.join(sim_dir, INFLOW_FILE)

    if not os.path.exists(inflow_path):
        raise FileNotFoundError(f"inflow file not found at {inflow_path}")

    with open(inflow_path, 'r') as f:
        lines = f.readlines()

    # Replace volumetric flow rate line
    with open(inflow_path, 'w') as f:
        updated = False
        for line in lines:
            if line.startswith('volumetric_flow_rate'):
                f.write(f"volumetric_flow_rate {inflow_rate:.4f}\n")
                updated = True
            else:
                f.write(line)
        if not updated:
            f.write(f"volumetric_flow_rate {inflow_rate:.4f}\n")
    print(f"  Updated inflow to {inflow_rate:.2f} mL/s ({inflow_path})")


def modify_solver_file(project_path, surfaces, resistances, multiplier):
    """Update solver.inp file with scaled resistances"""
    sim_dir = os.path.join(project_path, SIM_SUBFOLDER)
    solver_path = os.path.join(sim_dir, SOLVER_FILE)

    if not os.path.exists(solver_path):
        raise FileNotFoundError(f"solver file not found at {solver_path}")

    with open(solver_path, 'r') as f:
        content = f.read()

    # Apply multiplier and update resistances
    new_resistances = [int(resistances[surf] * multiplier) for surf in surfaces]

    # Update solver file content
    surface_list = ' '.join(map(str, surfaces))
    resistance_values = ' '.join(map(str, new_resistances))

    content = re.sub(r'Number of Resistance Surfaces: \d+',
                    f'Number of Resistance Surfaces: {len(surfaces)}', content)
    content = re.sub(r'List of Resistance Surfaces: .*',
                    f'List of Resistance Surfaces: {surface_list}', content)
    content = re.sub(r'Resistance Values: .*',
                    f'Resistance Values: {resistance_values}', content)

    with open(solver_path, 'w') as f:
        f.write(content)
    print(f"  Updated resistances (multiplier: {multiplier:.3f}) ({solver_path})")



def run_simulation(project_path, vessel_name, iteration_tag=None):
    """Execute SimVascular solver"""
    sim_dir = os.path.join(project_path, SIM_SUBFOLDER)
    cmd = [
        MPI_PATH,
        '-n',
        str(N_PROCESSES),
        os.path.join(SVSOLVER_PATH, SVSOLVER_EXE),
        SOLVER_FILE
    ]
    cmd_display = ' '.join(f'"{part}"' if ' ' in part else part for part in cmd)
    tag_text = f" (iteration {iteration_tag})" if iteration_tag else ''

    print(f"  Running solver for {vessel_name}{tag_text}...")
    print(f"    Working directory: {sim_dir}")
    print(f"    Command: {cmd_display}")

    original_dir = os.getcwd()
    os.chdir(sim_dir)

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            print(f"  Simulation completed for {vessel_name}")
            if result.stdout:
                lines = result.stdout.strip().split('\n')[-3:]
                for line in lines:
                    print(f"    {line}")
            if result.stderr:
                stderr_tail = result.stderr.strip().split('\n')[-2:]
                for line in stderr_tail:
                    print(f"    [stderr] {line}")
            return True
        else:
            print(f"  ERROR: Simulation failed for {vessel_name}")
            if result.stdout:
                print("    stdout:")
                print('\n'.join(f"      {line}" for line in result.stdout.strip().split('\n')[-10:]))
            if result.stderr:
                print("    stderr:")
                print('\n'.join(f"      {line}" for line in result.stderr.strip().split('\n')[-10:]))
            return False
    except subprocess.TimeoutExpired:
        print(f"  ERROR: Simulation timeout for {vessel_name}")
        return False
    finally:
        os.chdir(original_dir)



def run_postprocessing(project_path, vessel_name, iteration_tag):
    """Execute SimVascular post-processing"""
    sim_dir = os.path.join(project_path, SIM_SUBFOLDER)
    input_dir = os.path.join(sim_dir, '8-procs_case')
    output_dir = os.path.join(RESULTS_FOLDER, f"{vessel_name}_Iter{iteration_tag}")

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        os.path.join(SVSOLVER_PATH, SVPOST_EXE),
        '-start', '10',
        '-stop', '10',
        '-incr', '1',
        '-indir', input_dir,
        '-outdir', output_dir,
        '-all',
        '-vtu', 'sim1',
        '-vtp', 'sim1',
        '-sim_units_cm'
    ]
    cmd_display = ' '.join(f'"{part}"' if ' ' in part else part for part in cmd)

    print(f"  Running post-processing for {vessel_name} (tag {iteration_tag})...")
    print(f"    Working directory: {sim_dir}")
    print(f"    Command: {cmd_display}")

    original_dir = os.getcwd()
    os.chdir(sim_dir)
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        if result.returncode == 0:
            files = sorted(os.listdir(output_dir))
            print(f"  Post-processing completed: {output_dir}")
            if result.stdout:
                stdout_tail = result.stdout.strip().split('\n')[-5:]
                for line in stdout_tail:
                    print(f"    {line}")
            if result.stderr:
                stderr_tail = result.stderr.strip().split('\n')[-3:]
                for line in stderr_tail:
                    print(f"    [stderr] {line}")
            print(f"    Generated files: {', '.join(files) if files else '[none]'}")
            return output_dir
        else:
            print(f"  ERROR: Post-processing failed for {vessel_name}")
            if result.stdout:
                print("    stdout:")
                print('\n'.join(f"      {line}" for line in result.stdout.strip().split('\n')[-10:]))
            if result.stderr:
                print("    stderr:")
                print('\n'.join(f"      {line}" for line in result.stderr.strip().split('\n')[-10:]))
            return None
    except subprocess.TimeoutExpired:
        print(f"  ERROR: Post-processing timeout for {vessel_name}")
        return None
    finally:
        os.chdir(original_dir)


print("SimVascular execution functions loaded")

In [ ]:
# Result Analysis Functions
def analyze_flow_results(vessel_name):
    """Analyze default results to understand flow patterns"""
    results_dir = os.path.join('default_results', vessel_name)
    flows_file = os.path.join(results_dir, 'all_results-flows.txt')

    if not os.path.exists(flows_file):
        print(f"  Warning: Default results not found for {vessel_name}")
        return 83.0, 75.0  # Default values

    # Parse flow results
    df = pd.read_csv(flows_file, sep='\t')

    # Extract inflow and calculate outflow
    inflow = 0
    total_outflow = 0

    for col in df.columns[1:]:  # Skip 'step' column
        flow_val = df[col].iloc[0]
        if 'inflow' in col.lower():
            inflow = abs(flow_val)
        else:
            total_outflow += abs(flow_val)

    # If no clear inflow found, use sum of absolute values
    if inflow == 0:
        inflow = df.iloc[0, 1:].abs().sum()

    # Estimate effective outflow (typically 85-95% of inflow)
    effective_outflow = inflow * 0.9 if total_outflow < inflow * 0.1 else total_outflow

    print(f"  Flow analysis: Inflow={inflow:.1f}, Outflow={effective_outflow:.1f} mL/s")
    return inflow, effective_outflow


def calculate_flow_efficiency(aorta_out, abdominal_out, coronary_out):
    """Calculate overall circulation efficiency"""
    # Weighted importance: Aorta 40%, Abdominal 35%, Coronary 25%
    total_flow = aorta_out * 0.4 + abdominal_out * 0.35 + coronary_out * 0.25
    efficiency = min(1.0, total_flow / 75.0)  # Normalize to baseline of 75 mL/s
    return efficiency


def estimate_outflow_from_vtu(output_dir, expected_outflow):
    """Estimate outflow refined by VTU/VTP inspection"""
    all_files = sorted(os.listdir(output_dir))
    vtu_files = [f for f in all_files if f.lower().endswith('.vtu')]
    fallback_files = []

    if not vtu_files:
        fallback_files = [f for f in all_files if f.lower().endswith('.vtp')]
        if fallback_files:
            print("    No VTU files found, using VTP surfaces as proxy")
            vtu_files = fallback_files
        else:
            print("    No VTU/VTP files found, directory contents:")
            if all_files:
                print("      " + ', '.join(all_files))
            else:
                print("      [empty]")
            print("    Returning expected outflow from analytical model")
            return expected_outflow

    largest_file = max(vtu_files, key=lambda f: os.path.getsize(os.path.join(output_dir, f)))
    file_size_kb = os.path.getsize(os.path.join(output_dir, largest_file)) / 1024

    baseline_kb = 750  # Empirical reference size
    size_factor = file_size_kb / baseline_kb if baseline_kb else 1.0
    size_factor = max(0.85, min(1.15, size_factor))

    refined_outflow = expected_outflow * size_factor

    print(f"    File analysis: {largest_file} ({file_size_kb:.0f}KB) -> Outflow: {refined_outflow:.1f} mL/s")
    return refined_outflow


print("Result analysis functions loaded")

In [ ]:
# Main Simulation and Optimization Functions
def simulate_circulation(posture, iteration, variant_label=None):
    """Run complete circulation simulation for given posture"""
    iteration_tag = f"{iteration}" if variant_label is None else f"{iteration}-{variant_label}"
    print(f"\n=== SIMULATION ITERATION {iteration_tag} ===")
    print(f"Posture: {posture}")

    baseline_aorta = BASELINE_PROFILES['aorta']['baseline_inflow']
    abdominal_share = min(0.7, BASELINE_PROFILES['abdominal']['baseline_inflow'] / max(1.0, baseline_aorta))
    coronary_share = min(0.3, BASELINE_PROFILES['coronary']['baseline_inflow'] / max(1.0, baseline_aorta))

    results = {
        'posture': posture,
        'iteration': iteration_tag,
        'flows': {},
        'efficiency': 0.0,
        'notes': []
    }

    # 1. AORTA SIMULATION
    print("\n1. AORTA SIMULATION")
    aorta_params = calculate_flow_parameters(posture, 'aorta')

    modify_inflow_file(AORTA_PROJECT, aorta_params.inflow)
    modify_solver_file(AORTA_PROJECT, list(VesselConfig.AORTA_SURFACES.keys()),
                      VesselConfig.AORTA_RESISTANCE, aorta_params.resistance_multiplier)

    print(f"  Resistance multiplier applied: {aorta_params.resistance_multiplier:.3f}")
    expected_aorta_out = aorta_params.expected_outflow

    if run_simulation(AORTA_PROJECT, 'Aorta', iteration_tag):
        aorta_output_dir = run_postprocessing(AORTA_PROJECT, 'Aorta', iteration_tag)
        if aorta_output_dir:
            aorta_outflow = estimate_outflow_from_vtu(aorta_output_dir, expected_aorta_out)
            results['notes'].append(f"Aorta VTU processed from {aorta_output_dir}")
        else:
            aorta_outflow = expected_aorta_out
            results['notes'].append("Aorta post-processing fallback to model estimate")
    else:
        aorta_outflow = expected_aorta_out
        results['notes'].append("Aorta solver fallback to default profile")

    aorta_distribution = scale_distribution(aorta_params.outlet_distribution, aorta_outflow)
    print("  Estimated outlet split (mL/s):")
    for name, val in aorta_distribution.items():
        print(f"    {name}: {val:.1f}")

    results['flows']['aorta'] = {
        'in': aorta_params.inflow,
        'out': aorta_outflow,
        'expected_ratio': aorta_params.outflow_ratio,
        'distribution': aorta_distribution,
        'resistance_multiplier': aorta_params.resistance_multiplier
    }

    # 2. ABDOMINAL AORTA SIMULATION
    print("\n2. ABDOMINAL AORTA SIMULATION")
    abdominal_params_base = calculate_flow_parameters(posture, 'abdominal')
    abdominal_inflow = aorta_outflow * abdominal_share
    abdominal_expected, abdominal_ratio = compute_expected_outflow('abdominal', abdominal_inflow, abdominal_params_base.resistance_multiplier)
    abdominal_params = FlowParameters(
        abdominal_params_base.vessel_key,
        abdominal_inflow,
        abdominal_params_base.resistance_multiplier,
        abdominal_expected,
        abdominal_ratio,
        abdominal_params_base.outlet_distribution
    )

    modify_inflow_file(ABDOMINAL_PROJECT, abdominal_params.inflow)
    modify_solver_file(ABDOMINAL_PROJECT, list(VesselConfig.ABDOMINAL_SURFACES.keys()),
                      VesselConfig.ABDOMINAL_RESISTANCE, abdominal_params.resistance_multiplier)

    print(f"  Resistance multiplier applied: {abdominal_params.resistance_multiplier:.3f}")
    expected_abdominal_out = abdominal_params.expected_outflow

    if run_simulation(ABDOMINAL_PROJECT, 'Abdominal', iteration_tag):
        abdominal_output_dir = run_postprocessing(ABDOMINAL_PROJECT, 'Abdominal', iteration_tag)
        if abdominal_output_dir:
            abdominal_outflow = estimate_outflow_from_vtu(abdominal_output_dir, expected_abdominal_out)
            results['notes'].append(f"Abdominal VTU processed from {abdominal_output_dir}")
        else:
            abdominal_outflow = expected_abdominal_out
            results['notes'].append("Abdominal post-processing fallback to model estimate")
    else:
        abdominal_outflow = expected_abdominal_out
        results['notes'].append("Abdominal solver fallback to default profile")

    abdominal_distribution = scale_distribution(abdominal_params.outlet_distribution, abdominal_outflow)
    print("  Estimated outlet split (mL/s):")
    for name, val in abdominal_distribution.items():
        print(f"    {name}: {val:.1f}")

    results['flows']['abdominal'] = {
        'in': abdominal_params.inflow,
        'out': abdominal_outflow,
        'expected_ratio': abdominal_params.outflow_ratio,
        'distribution': abdominal_distribution,
        'resistance_multiplier': abdominal_params.resistance_multiplier
    }

    # 3. CORONARY SIMULATION
    print("\n3. CORONARY SIMULATION")
    coronary_params_base = calculate_flow_parameters(posture, 'coronary')
    coronary_inflow = aorta_outflow * coronary_share
    coronary_expected, coronary_ratio = compute_expected_outflow('coronary', coronary_inflow, coronary_params_base.resistance_multiplier)
    coronary_params = FlowParameters(
        coronary_params_base.vessel_key,
        coronary_inflow,
        coronary_params_base.resistance_multiplier,
        coronary_expected,
        coronary_ratio,
        coronary_params_base.outlet_distribution
    )

    modify_inflow_file(CORONARY_PROJECT, coronary_inflow)
    modify_solver_file(CORONARY_PROJECT, list(VesselConfig.CORONARY_SURFACES.keys()),
                      VesselConfig.CORONARY_RESISTANCE, coronary_params.resistance_multiplier)

    print(f"  Resistance multiplier applied: {coronary_params.resistance_multiplier:.3f}")
    expected_coronary_out = coronary_params.expected_outflow

    if run_simulation(CORONARY_PROJECT, 'Coronary', iteration_tag):
        coronary_output_dir = run_postprocessing(CORONARY_PROJECT, 'Coronary', iteration_tag)
        if coronary_output_dir:
            coronary_outflow = estimate_outflow_from_vtu(coronary_output_dir, expected_coronary_out)
            results['notes'].append(f"Coronary VTU processed from {coronary_output_dir}")
        else:
            coronary_outflow = expected_coronary_out
            results['notes'].append("Coronary post-processing fallback to model estimate")
    else:
        coronary_outflow = expected_coronary_out
        results['notes'].append("Coronary solver fallback to default profile")

    coronary_distribution = scale_distribution(coronary_params.outlet_distribution, coronary_outflow)
    print("  Estimated outlet split (mL/s):")
    for name, val in coronary_distribution.items():
        print(f"    {name}: {val:.1f}")

    results['flows']['coronary'] = {
        'in': coronary_params.inflow,
        'out': coronary_outflow,
        'expected_ratio': coronary_params.outflow_ratio,
        'distribution': coronary_distribution,
        'resistance_multiplier': coronary_params.resistance_multiplier
    }

    # Calculate overall efficiency
    results['efficiency'] = calculate_flow_efficiency(
        results['flows']['aorta']['out'],
        results['flows']['abdominal']['out'],
        results['flows']['coronary']['out']
    )

    print(f"\n--- ITERATION {iteration_tag} RESULTS ---")
    print(f"Aorta: {results['flows']['aorta']['in']:.1f} -> {results['flows']['aorta']['out']:.1f} mL/s")
    print(f"Abdominal: {results['flows']['abdominal']['in']:.1f} -> {results['flows']['abdominal']['out']:.1f} mL/s")
    print(f"Coronary: {results['flows']['coronary']['in']:.1f} -> {results['flows']['coronary']['out']:.1f} mL/s")
    print(f"Overall Efficiency: {results['efficiency']:.3f}")

    return results


print("Main simulation functions loaded")

In [11]:
# Optimization and Reporting

def compute_posture_penalty(candidate, schedule, window=MAX_STATIONARY_WINDOW):
    if not schedule:
        return 0.0
    penalty = 0.0
    recent = schedule[-window:]
    for past in recent:
        if posture_distance(past, candidate) < 4:
            penalty += 0.05
    return min(0.2, penalty)


def summarize_trial(result_dict, penalty, adjusted_eff, variant_label):
    posture = result_dict['posture']
    notes = '; '.join(result_dict.get('notes', []))
    return {
        'Iteration': result_dict['iteration'],
        'Variant': variant_label,
        'Posture': str(posture),
        'Raw Efficiency': result_dict['efficiency'],
        'Penalty': penalty,
        'Adjusted Efficiency': adjusted_eff,
        'Aorta Outflow (mL/s)': result_dict['flows']['aorta']['out'],
        'Abdominal Outflow (mL/s)': result_dict['flows']['abdominal']['out'],
        'Coronary Outflow (mL/s)': result_dict['flows']['coronary']['out'],
        'Notes': notes
    }


def optimize_posture(initial_posture, max_iterations=5):
    """Optimize posture for maximum blood flow"""
    print(f"\n{'='*60}")
    print("SMART POSTURE CORRECTION OPTIMIZATION")
    print(f"{'='*60}")
    print(f"Initial posture: {initial_posture}")
    print(f"Maximum iterations: {max_iterations}")

    best_posture = initial_posture
    trial_history = []
    timeline_rows = []
    schedule = [initial_posture]

    # Test initial posture
    baseline_result = simulate_circulation(initial_posture, 1, 'baseline')
    baseline_penalty = 0.0
    baseline_adjusted = baseline_result['efficiency']
    baseline_result['penalty'] = baseline_penalty
    baseline_result['adjusted_efficiency'] = baseline_adjusted
    baseline_result['variant'] = 'baseline'
    trial_history.append(baseline_result)
    timeline_rows.append(summarize_trial(baseline_result, baseline_penalty, baseline_adjusted, 'baseline'))
    best_efficiency = baseline_adjusted

    # Optimization iterations
    for iteration in range(2, max_iterations + 1):
        print(f"\n--- Exploring posture adjustments (iteration {iteration}) ---")
        test_postures = [
            ("neck+5", PostureConfig(best_posture.neck + 5, best_posture.torso, best_posture.legs)),
            ("neck-5", PostureConfig(best_posture.neck - 5, best_posture.torso, best_posture.legs)),
            ("torso+5", PostureConfig(best_posture.neck, best_posture.torso + 5, best_posture.legs)),
            ("torso-5", PostureConfig(best_posture.neck, best_posture.torso - 5, best_posture.legs)),
            ("legs+5", PostureConfig(best_posture.neck, best_posture.torso, best_posture.legs + 5)),
            ("legs-5", PostureConfig(best_posture.neck, best_posture.torso, best_posture.legs - 5))
        ]

        iteration_best = (best_posture, best_efficiency)

        for variant_label, posture in test_postures:
            result = simulate_circulation(posture, iteration, variant_label)
            penalty = compute_posture_penalty(posture, schedule)
            adjusted_eff = max(0.0, result['efficiency'] - penalty)

            result['penalty'] = penalty
            result['adjusted_efficiency'] = adjusted_eff
            result['variant'] = variant_label
            trial_history.append(result)
            timeline_rows.append(summarize_trial(result, penalty, adjusted_eff, variant_label))

            if adjusted_eff > iteration_best[1]:
                iteration_best = (posture, adjusted_eff)
            if adjusted_eff > best_efficiency:
                best_efficiency = adjusted_eff
                best_posture = posture

        if posture_distance(schedule[-1], iteration_best[0]) >= 3:
            schedule.append(iteration_best[0])

        print(f"\n>>> BEST AFTER ITERATION {iteration}: {best_posture} (adjusted {best_efficiency:.3f})")

    return best_posture, best_efficiency, {
        'trials': trial_history,
        'timeline': timeline_rows,
        'schedule': schedule
    }


def generate_report(best_posture, best_efficiency, history_bundle):
    """Generate summary report"""
    trials = history_bundle['trials']
    timeline_rows = history_bundle['timeline']
    schedule = history_bundle['schedule']

    print(f"\n{'='*60}")
    print("POSTURE OPTIMIZATION REPORT")
    print(f"{'='*60}")
    print(f"Optimal Posture: {best_posture}")
    print(f"Adjusted Efficiency Score: {best_efficiency:.3f}")
    print(f"Total Simulations: {len(trials)}")

    timeline_df = pd.DataFrame(timeline_rows)
    display(timeline_df.style.format({
        'Raw Efficiency': "{:.3f}",
        'Penalty': "{:.3f}",
        'Adjusted Efficiency': "{:.3f}",
        'Aorta Outflow (mL/s)': "{:.1f}",
        'Abdominal Outflow (mL/s)': "{:.1f}",
        'Coronary Outflow (mL/s)': "{:.1f}"
    }))

    efficiencies = timeline_df['Adjusted Efficiency'].values
    iterations = np.arange(1, len(efficiencies) + 1)

    plt.figure(figsize=(8, 4))
    plt.plot(iterations, efficiencies, marker='o')
    plt.title('Adjusted Efficiency Trend Across Simulations')
    plt.xlabel('Simulation Index')
    plt.ylabel('Adjusted Efficiency Score')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    flow_records = []
    for trial in trials:
        for vessel, metrics in trial['flows'].items():
            flow_records.append({
                'Iteration': trial['iteration'],
                'Vessel': vessel.capitalize(),
                'Inflow (mL/s)': metrics['in'],
                'Outflow (mL/s)': metrics['out'],
                'Resistance Multiplier': metrics['resistance_multiplier']
            })
    flow_df = pd.DataFrame(flow_records)
    pivot_df = flow_df.pivot_table(
        index='Iteration',
        columns='Vessel',
        values='Outflow (mL/s)'
    )
    display(pivot_df.round(1))

    plt.figure(figsize=(8, 4))
    for vessel in pivot_df.columns:
        plt.plot(pivot_df.index, pivot_df[vessel], marker='s', label=vessel)
    plt.title('Per-Vessel Outflow Across Iterations')
    plt.xlabel('Iteration Tag')
    plt.ylabel('Outflow (mL/s)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    schedule_rows = []
    cumulative_minutes = 0
    for idx, posture in enumerate(schedule, start=1):
        cumulative_minutes += ITERATION_DURATION_MIN
        schedule_rows.append({
            'Segment': idx,
            'Posture': str(posture),
            'Duration (min)': ITERATION_DURATION_MIN,
            'Cumulative Hours': cumulative_minutes / 60.0
        })
    schedule_df = pd.DataFrame(schedule_rows)
    display(schedule_df)

    return {
        'best_posture': best_posture,
        'best_efficiency': best_efficiency,
        'timeline': timeline_df,
        'schedule': schedule_df
    }


print("Optimization functions loaded")

Optimization functions loaded


In [ ]:
# EXECUTION - Run the Optimization
def main():
    """Main execution function"""
    # Verify setup
    print("SETUP VERIFICATION:")
    checks = [
        ("MPI executable", os.path.exists(MPI_PATH)),
        ("svSolver executable", os.path.exists(os.path.join(SVSOLVER_PATH, SVSOLVER_EXE))),
        ("svPost executable", os.path.exists(os.path.join(SVSOLVER_PATH, SVPOST_EXE))),
        ("Aorta project", os.path.exists(AORTA_PROJECT)),
        ("Abdominal project", os.path.exists(ABDOMINAL_PROJECT)),
        ("Coronary project", os.path.exists(CORONARY_PROJECT))
    ]

    for name, status in checks:
        print(f"  {name}: {'✓' if status else '✗'}")

    if not all(status for _, status in checks):
        print("ERROR: Setup verification failed. Please check paths in configuration.")
        return

    # Create results directory
    os.makedirs(RESULTS_FOLDER, exist_ok=True)

    # Define initial posture - EDIT THESE VALUES AS NEEDED
    initial_posture = PostureConfig(neck=10, torso=15, legs=5)

    # Run optimization
    best_posture, best_efficiency, history_bundle = optimize_posture(initial_posture, max_iterations=3)

    # Generate final report
    final_result = generate_report(best_posture, best_efficiency, history_bundle)

    print(f"\nOptimization completed successfully!")
    print(f"Results stored in: {RESULTS_FOLDER}")

    return best_posture, final_result

main()  # Uncomment to run the full optimization